In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost

import os
import glob
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
DATA_BASE = os.path.join(PROJECT_DIR, 'data/raw')
RANDOM_STATE = 42

Mounted at /content/drive


In [2]:
def compare_with_without_duplicates(name, df, label_col, drop_cols=None):
    print(f"\n{'='*70}\n🔎 {name}\n{'='*70}")
    drop_cols = drop_cols or []
    df = df.copy()

    print(f"Початковий розмір: {df.shape[0]} рядків, {df.shape[1]} колонок")

                                                                  
    n_dup_full = df.duplicated(keep=False).sum()
    print(f"\nПовністю ідентичних рядків (усі колонки+мітка): {n_dup_full} "
          f"({100*n_dup_full/len(df):.2f}%)")

                                                                             
    feature_cols = [c for c in df.columns if c not in drop_cols + [label_col]]
    n_dup_features_only = df.duplicated(subset=feature_cols, keep=False).sum()
    print(f"Ідентичних за ОЗНАКАМИ (без мітки, без службових колонок): "
          f"{n_dup_features_only} ({100*n_dup_features_only/len(df):.2f}%)")

                                                                              
    if n_dup_features_only > 0:
        dup_mask = df.duplicated(subset=feature_cols, keep=False)
        conflicting = df[dup_mask].groupby(feature_cols)[label_col].nunique()
        n_conflicting_groups = (conflicting > 1).sum()
        if n_conflicting_groups > 0:
            print(f"⚠️ УВАГА: {n_conflicting_groups} груп ідентичних ознак мають "
                  f"РІЗНУ мітку (суперечливі дані) — модель у принципі не може "
                  f"правильно класифікувати обидві копії одночасно.")

                                                                 
    df_dedup = df.drop_duplicates(subset=feature_cols + [label_col]).reset_index(drop=True)
    print(f"\nПісля видалення повних дублікатів (ознаки+мітка): "
          f"{df_dedup.shape[0]} рядків (було {df.shape[0]}, "
          f"видалено {df.shape[0]-df_dedup.shape[0]})")

    def train_eval(data, tag):
        X = data[feature_cols].select_dtypes(include=[np.number])
        y = data[label_col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )
        model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                        class_weight='balanced', n_jobs=-1)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        print(f"  [{tag}] n={len(data):>7}  Acc={acc:.4f}  F1={f1:.4f}  "
              f"Prec={prec:.4f}  Rec={rec:.4f}")
        return {'tag': tag, 'n': len(data), 'accuracy': acc, 'f1': f1,
                'precision': prec, 'recall': rec}

    print("\n--- Порівняння якості RandomForest ---")
    res_with = train_eval(df, "З дублікатами (як у 03)")
    res_without = train_eval(df_dedup, "Без дублікатів (дедупліковано)")

    drop_f1 = res_with['f1'] - res_without['f1']
    print(f"\n📊 Різниця F1 (з дублікатами − без дублікатів): {drop_f1:+.4f}")
    if drop_f1 > 0.02:
        print("   ⚠️ Суттєве падіння F1 після дедуплікації — дублікати справді "
              "роздували внутрішню точність у 03. Рекомендується прибрати "
              "дублікати перед фінальним навчанням.")
    elif drop_f1 > 0.005:
        print("   ℹ️ Невелике падіння F1 — дублікати мали помірний вплив.")
    else:
        print("   ✅ F1 практично не змінився — дублікати НЕ є основною причиною "
              "високої внутрішньої точності на цьому датасеті.")

    return {'dataset': name, 'n_full_dup': int(n_dup_full),
            'n_feature_dup': int(n_dup_features_only),
            'f1_with_dup': res_with['f1'], 'f1_without_dup': res_without['f1'],
            'f1_drop': drop_f1}

In [3]:
cic_files = glob.glob(os.path.join(DATA_BASE, 'cic_ids_2017', '*.csv'))
df_cic = pd.concat([pd.read_csv(f, low_memory=False) for f in cic_files], ignore_index=True)
df_cic = df_cic.replace([np.inf, -np.inf], np.nan).dropna()
df_cic['label_bin'] = (df_cic[' Label'].astype(str).str.strip() != 'BENIGN').astype(int)

result_cic = compare_with_without_duplicates(
    "CIC-IDS2017", df_cic, label_col='label_bin', drop_cols=[' Label']
)


🔎 CIC-IDS2017
Початковий розмір: 1108028 рядків, 80 колонок

Повністю ідентичних рядків (усі колонки+мітка): 118752 (10.72%)
Ідентичних за ОЗНАКАМИ (без мітки, без службових колонок): 118826 (10.72%)
⚠️ УВАГА: 61 груп ідентичних ознак мають РІЗНУ мітку (суперечливі дані) — модель у принципі не може правильно класифікувати обидві копії одночасно.

Після видалення повних дублікатів (ознаки+мітка): 1007649 рядків (було 1108028, видалено 100379)

--- Порівняння якості RandomForest ---
  [З дублікатами (як у 03)] n=1108028  Acc=0.9993  F1=0.9990  Prec=0.9989  Rec=0.9990
  [Без дублікатів (дедупліковано)] n=1007649  Acc=0.9990  F1=0.9985  Prec=0.9987  Rec=0.9982

📊 Різниця F1 (з дублікатами − без дублікатів): +0.0005
   ✅ F1 практично не змінився — дублікати НЕ є основною причиною високої внутрішньої точності на цьому датасеті.


In [4]:
df_unsw_train = pd.read_csv(os.path.join(DATA_BASE, 'unsw_nb15', 'UNSW_NB15_training-set.csv'), low_memory=False)
df_unsw_test = pd.read_csv(os.path.join(DATA_BASE, 'unsw_nb15', 'UNSW_NB15_testing-set.csv'), low_memory=False)
df_unsw = pd.concat([df_unsw_train, df_unsw_test], ignore_index=True)
df_unsw = df_unsw.replace([np.inf, -np.inf], np.nan).dropna()

result_unsw = compare_with_without_duplicates(
    "UNSW-NB15", df_unsw, label_col='label', drop_cols=['id', 'attack_cat']
)


🔎 UNSW-NB15
Початковий розмір: 257673 рядків, 45 колонок

Повністю ідентичних рядків (усі колонки+мітка): 0 (0.00%)
Ідентичних за ОЗНАКАМИ (без мітки, без службових колонок): 119701 (46.45%)
⚠️ УВАГА: 414 груп ідентичних ознак мають РІЗНУ мітку (суперечливі дані) — модель у принципі не може правильно класифікувати обидві копії одночасно.

Після видалення повних дублікатів (ознаки+мітка): 154098 рядків (було 257673, видалено 103575)

--- Порівняння якості RandomForest ---
  [З дублікатами (як у 03)] n= 257673  Acc=0.9521  F1=0.9624  Prec=0.9640  Rec=0.9609
  [Без дублікатів (дедупліковано)] n= 154098  Acc=0.9172  F1=0.9059  Prec=0.9142  Rec=0.8977

📊 Різниця F1 (з дублікатами − без дублікатів): +0.0566
   ⚠️ Суттєве падіння F1 після дедуплікації — дублікати справді роздували внутрішню точність у 03. Рекомендується прибрати дублікати перед фінальним навчанням.


In [5]:
df_ctu_attack = pd.read_csv(os.path.join(DATA_BASE, 'ctu13', 'CTU13_Attack_Traffic.csv'), low_memory=False)
df_ctu_normal = pd.read_csv(os.path.join(DATA_BASE, 'ctu13', 'CTU13_Normal_Traffic.csv'), low_memory=False)
raw_label_variants = ['Label', 'LABEL', 'Label_original']
df_ctu_attack = df_ctu_attack.drop(columns=[c for c in raw_label_variants if c in df_ctu_attack.columns], errors='ignore')
df_ctu_normal = df_ctu_normal.drop(columns=[c for c in raw_label_variants if c in df_ctu_normal.columns], errors='ignore')
common_cols = df_ctu_attack.columns.intersection(df_ctu_normal.columns)
df_ctu_attack, df_ctu_normal = df_ctu_attack[common_cols].copy(), df_ctu_normal[common_cols].copy()
df_ctu_attack['label'] = 1
df_ctu_normal['label'] = 0
df_ctu = pd.concat([df_ctu_attack, df_ctu_normal], ignore_index=True)
df_ctu = df_ctu.replace([np.inf, -np.inf], np.nan).dropna()

result_ctu = compare_with_without_duplicates(
    "CTU-13", df_ctu, label_col='label', drop_cols=['Unnamed: 0']
)


🔎 CTU-13
Початковий розмір: 92212 рядків, 59 колонок

Повністю ідентичних рядків (усі колонки+мітка): 0 (0.00%)
Ідентичних за ОЗНАКАМИ (без мітки, без службових колонок): 11140 (12.08%)
⚠️ УВАГА: 1 груп ідентичних ознак мають РІЗНУ мітку (суперечливі дані) — модель у принципі не може правильно класифікувати обидві копії одночасно.

Після видалення повних дублікатів (ознаки+мітка): 82721 рядків (було 92212, видалено 9491)

--- Порівняння якості RandomForest ---
  [З дублікатами (як у 03)] n=  92212  Acc=0.9976  F1=0.9972  Prec=0.9968  Rec=0.9976
  [Без дублікатів (дедупліковано)] n=  82721  Acc=0.9970  F1=0.9960  Prec=0.9955  Rec=0.9966

📊 Різниця F1 (з дублікатами − без дублікатів): +0.0012
   ✅ F1 практично не змінився — дублікати НЕ є основною причиною високої внутрішньої точності на цьому датасеті.


In [6]:
summary = pd.DataFrame([result_cic, result_unsw, result_ctu])
print("="*70)
print("ЗВЕДЕНИЙ ПІДСУМОК — Питання №4 (дублікати рядків)")
print("="*70)
print(summary.to_string(index=False))

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'duplicate_check_summary.csv'), index=False)

print("\n📌 Рекомендація для тексту роботи:")
for _, row in summary.iterrows():
    if row['f1_drop'] > 0.02:
        verdict = "дублікати ІСТОТНО впливають на внутрішню точність — потрібна дедуплікація перед фінальним навчанням."
    elif row['f1_drop'] > 0.005:
        verdict = "дублікати мають ПОМІРНИЙ вплив — варто згадати як обмеження, дедуплікація на розсуд."
    else:
        verdict = "дублікати НЕ є значущою причиною високої точності — можна прибрати це застереження з обмежень."
    print(f"  {row['dataset']}: {verdict}")

print(f"\n✅ Таблицю збережено у "
      f"{os.path.join(OUTPUT_DIR, 'tables', 'duplicate_check_summary.csv')}")

ЗВЕДЕНИЙ ПІДСУМОК — Питання №4 (дублікати рядків)
    dataset  n_full_dup  n_feature_dup  f1_with_dup  f1_without_dup  f1_drop
CIC-IDS2017      118752         118826     0.998952        0.998478 0.000474
  UNSW-NB15           0         119701     0.962443        0.905878 0.056565
     CTU-13           0          11140     0.997173        0.996019 0.001155

📌 Рекомендація для тексту роботи:
  CIC-IDS2017: дублікати НЕ є значущою причиною високої точності — можна прибрати це застереження з обмежень.
  UNSW-NB15: дублікати ІСТОТНО впливають на внутрішню точність — потрібна дедуплікація перед фінальним навчанням.
  CTU-13: дублікати НЕ є значущою причиною високої точності — можна прибрати це застереження з обмежень.

✅ Таблицю збережено у /content/drive/MyDrive/man-bot-detection/notebooks/results/tables/duplicate_check_summary.csv
